In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


In [2]:

# Define paths
train_dir = 'Training/'
test_dir = 'Testing/'
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']


In [3]:

# Function to load images and labels
def load_images_and_labels(data_dir, classes, image_size=(128, 128)):
    X = []
    y = []
    for i, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        for img_name in os.listdir(cls_dir):
            img_path = os.path.join(cls_dir, img_name)
            img = Image.open(img_path).convert('L')  # Convert to grayscale
            img = img.resize(image_size)  # Resize to fixed size
            img_array = np.array(img).flatten()  # Flatten the image
            X.append(img_array)
            y.append(i)
    return np.array(X), np.array(y)

In [4]:
# Load training set
print("Loading training set...")
X_train, y_train = load_images_and_labels(train_dir, classes, image_size=(128, 128))
print(f"Training set loaded: {X_train.shape[0]} samples, {X_train.shape[1]} features")


Loading training set...
Training set loaded: 5600 samples, 16384 features


In [5]:
# Apply Principal Component Analysis (PCA)
print("Applying PCA...")
n_components = 100  # You can adjust this number based on your needs
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train)
print(f"PCA applied: reduced to {n_components} components")

# Train Support Vector Machine (SVM)
print("Training SVM...")
svm = SVC(kernel='rbf', C=1.0, gamma='scale')  # You can tune these parameters
svm.fit(X_train_pca, y_train)
print("SVM trained successfully")


Applying PCA...
PCA applied: reduced to 100 components
Training SVM...
SVM trained successfully


In [6]:

# Load testing set for evaluation
print("Loading testing set...")
X_test, y_test = load_images_and_labels(test_dir, classes, image_size=(128, 128))
print(f"Testing set loaded: {X_test.shape[0]} samples")

# Apply PCA to test set
X_test_pca = pca.transform(X_test)

# Predict and evaluate
print("Evaluating on test set...")
y_pred = svm.predict(X_test_pca)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Optional: Print classification report
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=classes))

Loading testing set...
Testing set loaded: 1600 samples
Evaluating on test set...
Test Accuracy: 0.8219

Classification Report:
              precision    recall  f1-score   support

      glioma       0.80      0.65      0.72       400
  meningioma       0.78      0.73      0.75       400
     notumor       0.82      0.96      0.88       400
   pituitary       0.88      0.95      0.91       400

    accuracy                           0.82      1600
   macro avg       0.82      0.82      0.82      1600
weighted avg       0.82      0.82      0.82      1600

